# Auxiliary / smoothed-tau target: fix the label noise near the break

Our training label is a HARD step: `y[t] = 1` the instant `t >= tau`. But agents.md §1 says a
break "may take some observations after the true break point to become detectable." So right
after `tau`, the series still LOOKS unbroken, yet the label already says 1. We are forcing the
model to call "broken" when no signal has emerged — **label noise in the just-after-break band**.
This is independent of every feature/localization lever (it touches the TARGET, not the inputs),
so it is a clean candidate to STACK.

The TS-AUC metric still scores the hard label — but a model trained to stay low until the signal
actually emerges should rank BETTER, because ranking a series high right after `tau` (no signal
yet) is a mistake the hard target actively encourages.

Variants (same 50 features, same folds, pointwise CatBoost), paired vs the hard-label base:
- **base** hard 0/1 label, classifier (= submit#3 training)
- **ramp** soft label: 0 before `tau`, ramps 0->1 over `[tau, tau+2W]`, 1 after — regressor
- **downweight** hard label, but sample_weight decayed in the `[tau, tau+2W]` band

Believe a win only at t >= 3 (5-fold t~2 is not safe — [[honest-cv-harness]]). If promising,
re-run at 10 folds and fold into the stack.

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor

TOOLS = os.path.abspath('../tools')
sys.path.insert(0, TOOLS)
import cv_tools as CV

cache = np.load(os.path.join(TOOLS, 'cv_folds_by_series', 'cv_cache_full.npz'))
X, y, sid, tim = cache['X'], cache['y'], cache['sid'], cache['time']
sampled = cache['is_sampled_step']
index = pd.MultiIndex.from_arrays([sid, tim], names=['id', 'time'])
step = CV.steps_from_index(index)

# tau_index per series -> per-row distance since break (NaN/inf for no-break)
idx = pd.read_parquet(os.path.join(TOOLS, '..', '..', '..', 'y_train_index.parquet'))
tau_of = idx['tau_index'].reindex(np.unique(sid))            # -1 = no break
tau_row = tau_of.reindex(sid).to_numpy()
since = step - tau_row                                       # steps since break; <0 = pre-break
is_break = tau_row >= 0
# sanity: hard label == (break series AND since>=0)
recon = (is_break & (since >= 0)).astype(np.int8)
print('hard-label reconstruction matches cache y:', bool((recon == y).all()))
print(f'{len(y):,} rows | break rows {is_break.sum():,} | pos {y.mean():.3f}')

hard-label reconstruction matches cache y: True
5,036,517 rows | break rows 2,516,113 | pos 0.256


In [2]:
W = 15   # detection-lag half-band: ramp spans [tau, tau+2W] = 30 steps

# soft ramp label in [0,1]: 0 pre-break, linear 0->1 across [tau, tau+2W], 1 after.
ramp = np.clip(since / (2.0 * W), 0.0, 1.0)
ramp[~is_break] = 0.0                                        # no-break stays 0
ramp[is_break & (since < 0)] = 0.0                          # pre-break stays 0

# near-break down-weight: rows in [tau, tau+2W] are the ambiguous ones -> shrink their weight.
band = is_break & (since >= 0) & (since < 2 * W)
dw = np.ones(len(y), np.float64)
dw[band] = np.clip(since[band] / (2.0 * W), 0.05, 1.0)      # 0.05 at tau -> 1.0 at tau+2W

print(f'ramp: {(ramp>0).sum():,} rows >0, {((ramp>0)&(ramp<1)).sum():,} in-ramp')
print(f'downweight: {band.sum():,} band rows, mean w {dw[band].mean():.3f}')

ramp: 1,283,079 rows >0, 134,306 in-ramp
downweight: 139,273 band rows, mean w 0.475


In [3]:
CLS = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
           bootstrap_type='Bernoulli', subsample=0.8, loss_function='Logloss',
           random_seed=0, verbose=0, thread_count=-1)
REG = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
           bootstrap_type='Bernoulli', subsample=0.8, loss_function='RMSE',
           random_seed=0, verbose=0, thread_count=-1)
folds = CV.series_folds(sid, n_splits=5, seed=0)

def fp_base(train, val):
    m = CatBoostClassifier(**CLS); m.fit(train.X, train.y, sample_weight=train.w)
    return m.predict_proba(val.X)[:, 1]

# run_cv builds Split internally, so route the custom targets/weights via row-index lookup.
ramp_by_row = pd.Series(ramp, index=index)
dw_by_row = pd.Series(dw, index=index)

def fp_ramp(train, val):
    yt = ramp_by_row.loc[train.index].to_numpy()
    m = CatBoostRegressor(**REG); m.fit(train.X, yt, sample_weight=train.w)
    return m.predict(val.X)

def fp_dw(train, val):
    w = train.w * dw_by_row.loc[train.index].to_numpy(); w = w / w.mean()
    m = CatBoostClassifier(**CLS); m.fit(train.X, train.y, sample_weight=w)
    return m.predict_proba(val.X)[:, 1]

res = {}
for name, fp in [('base', fp_base), ('ramp', fp_ramp), ('downweight', fp_dw)]:
    t = time.time(); print(f'\n=== {name} ===', flush=True)
    res[name] = CV.run_cv(X, y, sid, index, fp, folds=folds,
                          train_row_mask=sampled, row_step=step)
    json.dump({k: dict(fold_scores=v.fold_scores.tolist(), mean=v.mean, oof=v.oof_score)
               for k, v in res.items()}, open('smoothed_results.json', 'w'), indent=2)
    print(f'{res[name].summary(name)} | {time.time()-t:.0f}s', flush=True)


=== base ===


  fold 0: train   263,351 rows / 8,000 series | val 1,012,996 rows / 2,000 series | TS-AUC 0.5849


  fold 1: train   263,306 rows / 8,000 series | val   982,164 rows / 2,000 series | TS-AUC 0.5964


  fold 2: train   263,072 rows / 8,000 series | val 1,018,819 rows / 2,000 series | TS-AUC 0.5828


  fold 3: train   263,146 rows / 8,000 series | val 1,015,051 rows / 2,000 series | TS-AUC 0.6014


...

  fold 1: train   263,306 rows / 8,000 series | val   982,164 rows / 2,000 series | TS-AUC 0.6025


  fold 2: train   263,072 rows / 8,000 series | val 1,018,819 rows / 2,000 series | TS-AUC 0.5865


  fold 3: train   263,146 rows / 8,000 series | val 1,015,051 rows / 2,000 series | TS-AUC 0.6086


  fold 4: train   263,109 rows / 8,000 series | val 1,007,487 rows / 2,000 series | TS-AUC 0.6262


downweight         mean 0.6024 +/- 0.0162 (sem 0.0073) | OOF 0.6015 | folds: 0.5883  0.6025  0.5865  0.6086  0.6262 | 251s


In [4]:
print(CV.summarize(res))
print('\n--- soft-target variants vs hard-label base ---')
for n in ['ramp', 'downweight']:
    print(CV.paired_compare(res[n], res['base'], n, 'base'))

ramp               mean 0.6073 +/- 0.0149 (sem 0.0067) | OOF 0.6064 | folds: 0.5934  0.6027  0.5949  0.6176  0.6277
downweight         mean 0.6024 +/- 0.0162 (sem 0.0073) | OOF 0.6015 | folds: 0.5883  0.6025  0.5865  0.6086  0.6262
base               mean 0.5977 +/- 0.0162 (sem 0.0073) | OOF 0.5968 | folds: 0.5849  0.5964  0.5828  0.6014  0.6232

--- soft-target variants vs hard-label base ---
ramp - base: +0.0095 +/- 0.0047 (sem 0.0021, t +4.52, wins 5/5) -> consistent
  per-fold deltas: +0.0085  +0.0064  +0.0121  +0.0162  +0.0045
downweight - base: +0.0047 +/- 0.0019 (sem 0.0008, t +5.55, wins 5/5) -> consistent
  per-fold deltas: +0.0034  +0.0061  +0.0037  +0.0073  +0.0029
